In [1]:
!pip install transformers sentencepiece datasets -q

import pandas as pd
import numpy as np
import re
import torch
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from datasets import Dataset
from google.colab import drive

In [2]:
drive.mount('/content/drive')

CSV_PATH    = '/content/drive/MyDrive/Licenta/medical_dataset_final.csv'

Mounted at /content/drive


In [3]:
df = pd.read_csv(CSV_PATH, sep=';', on_bad_lines='skip')
df = df[['filename', 'rezumat', 'keywords']].copy()
df = df.dropna(subset=['rezumat', 'keywords'])
df['rezumat']  = df['rezumat'].str.strip()
df['keywords'] = df['keywords'].str.strip()

print(f"Total articole: {len(df)}")

Total articole: 1114


In [4]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"Train : {len(train_df)} articole")
print(f"Val   : {len(val_df)} articole")
print(f"Test  : {len(test_df)} articole")

Train : 779 articole
Val   : 167 articole
Test  : 168 articole


In [5]:
MODEL_NAME = 'google/mt5-base'
print(f"\nSe încarcă tokenizer-ul {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LEN  = 512   # lungime maximă rezumat
MAX_TARGET_LEN = 64    # lungime maximă keywords generate

PREFIX = "extrage cuvinte cheie medicale: "  # prefix pentru mt5


Se încarcă tokenizer-ul google/mt5-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [6]:
def preprocess(examples):
    inputs  = [PREFIX + text for text in examples['rezumat']]
    targets = examples['keywords']

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Conversie la HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['rezumat', 'keywords']].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[['rezumat', 'keywords']].reset_index(drop=True))

print("Tokenizez datele...")
train_dataset = train_dataset.map(preprocess, batched=True, remove_columns=['rezumat', 'keywords'])
val_dataset   = val_dataset.map(preprocess,   batched=True, remove_columns=['rezumat', 'keywords'])
print("Tokenizare completă.")

Tokenizez datele...


Map:   0%|          | 0/779 [00:00<?, ? examples/s]

Map:   0%|          | 0/167 [00:00<?, ? examples/s]

Tokenizare completă.


In [7]:
print(f"\nSe încarcă modelul {MODEL_NAME}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.config.tie_word_embeddings = False
model.config.decoder_start_token_id = tokenizer.pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
print(f"Parametri model: {model.num_parameters():,}")

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)


Se încarcă modelul google/mt5-base...


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Parametri model: 582,401,280


In [8]:
training_args = Seq2SeqTrainingArguments(
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=5e-4,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=50,
    save_total_limit=2,
    fp16=False,  # activează half-precision dacă e GPU
    report_to='none'
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [9]:
print("\nÎncepe antrenarea...")
trainer.train()
print("Antrenare completă!")


Începe antrenarea...


Epoch,Training Loss,Validation Loss
1,3.317364,2.682483
2,2.431596,2.187409
3,1.726415,2.164285
4,1.414922,2.219587
5,1.091070,2.313430
6,0.825388,2.394799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Antrenare completă!


In [13]:
SAVE_PATH = '/content/drive/MyDrive/Licenta/mt5_base_model'

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model salvat în {SAVE_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model salvat în /content/drive/MyDrive/Licenta/mt5_base_model


In [10]:
print("\nGenerez keywords pentru setul de test...")

def generate_keywords(text, num_beams=4):
    input_text = PREFIX + text
    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        max_length=MAX_INPUT_LEN,
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=MAX_TARGET_LEN,
        num_beams=num_beams,
        early_stopping=True,
        no_repeat_ngram_size=2   # evită repetiții
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

model.eval()
results = []
for _, row in test_df.iterrows():
    generated = generate_keywords(row['rezumat'])
    results.append({
        'filename'           : row['filename'],
        'keywords_generated' : generated,
        'keywords_original'  : row['keywords'],
    })

results_df = pd.DataFrame(results)


Generez keywords pentru setul de test...


In [11]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer, util as st_util

SIM_THRESHOLD = 0.6
print("\nSe încarcă modelul semantic pentru evaluare...")
sem_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def parse_keywords(text):
    if pd.isna(text) or str(text).strip() == '':
        return []
    return [w.strip().lower() for w in re.split(r'[;,]', str(text)) if w.strip()]

def compute_metrics(extracted_str, original_str):
    extracted = parse_keywords(extracted_str)
    original  = parse_keywords(original_str)
    if not extracted or not original:
        return 0.0, 0.0, 0.0

    all_terms  = extracted + original
    embeddings = sem_model.encode(all_terms, convert_to_tensor=True)
    emb_ext    = embeddings[:len(extracted)]
    emb_orig   = embeddings[len(extracted):]

    matched_orig = set()
    tp = 0
    for i, e in enumerate(extracted):
        for j, o in enumerate(original):
            if j in matched_orig:
                continue
            string_match   = e in o or o in e
            cos_score      = st_util.cos_sim(emb_ext[i], emb_orig[j]).item()
            semantic_match = cos_score >= SIM_THRESHOLD
            if string_match or semantic_match:
                tp += 1
                matched_orig.add(j)
                break

    precision = tp / len(extracted)
    recall    = tp / len(original)
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

print("Calculez metrici semantice...")
results_df[['precision', 'recall', 'f1']] = results_df.apply(
    lambda r: pd.Series(compute_metrics(r['keywords_generated'], r['keywords_original'])),
    axis=1
)

avg_p  = results_df['precision'].mean()
avg_r  = results_df['recall'].mean()
avg_f1 = results_df['f1'].mean()

print(f"\n{'='*45}")
print(f"  Precision medie : {avg_p:.4f}  ({avg_p:.2%})")
print(f"  Recall mediu    : {avg_r:.4f}  ({avg_r:.2%})")
print(f"  F1 mediu        : {avg_f1:.4f}  ({avg_f1:.2%})")
print(f"{'='*45}")
print(f"  Documente cu F1 > 0 : {(results_df['f1'] > 0).sum()} / {len(results_df)}")


Se încarcă modelul semantic pentru evaluare...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Calculez metrici semantice...

  Precision medie : 0.6169  (61.69%)
  Recall mediu    : 0.4709  (47.09%)
  F1 mediu        : 0.5151  (51.51%)
  Documente cu F1 > 0 : 159 / 168


In [12]:
from IPython.display import display, HTML

N_PREVIEW = 20

def make_html_table(df_preview):
    rows = ''
    for _, r in df_preview.iterrows():
        color = '#c8f7c5' if r['f1'] >= 0.5 else '#fde8d8' if r['f1'] > 0 else '#f5f5f5'
        rows += f"""
        <tr>
            <td style='font-size:12px;color:#555;padding:6px 8px;'>{r['filename']}</td>
            <td style='color:#6a0dad;font-weight:500;padding:6px 8px;'>{r['keywords_generated']}</td>
            <td style='color:#555;padding:6px 8px;'>{r['keywords_original']}</td>
            <td style='text-align:center;padding:6px 8px;'>{r['precision']:.2f}</td>
            <td style='text-align:center;padding:6px 8px;'>{r['recall']:.2f}</td>
            <td style='text-align:center;padding:6px 8px;background:{color};font-weight:bold;'>{r['f1']:.2f}</td>
        </tr>"""

    html = f"""
    <h3 style='font-family:sans-serif;'>🤖 mT5 — Primele {N_PREVIEW} articole (set de test)</h3>
    <table style='border-collapse:collapse;width:100%;font-family:sans-serif;font-size:13px;'>
        <thead>
            <tr style='background:#6a0dad;color:white;'>
                <th style='padding:8px;text-align:left;'>Fișier</th>
                <th style='padding:8px;text-align:left;'>Keywords generate (mT5)</th>
                <th style='padding:8px;text-align:left;'>Keywords originale</th>
                <th style='padding:8px;text-align:center;'>Precision</th>
                <th style='padding:8px;text-align:center;'>Recall</th>
                <th style='padding:8px;text-align:center;'>F1</th>
            </tr>
        </thead>
        <tbody>{rows}</tbody>
    </table>
    <p style='font-family:sans-serif;font-size:12px;color:#888;margin-top:8px;'>
        🟢 F1 ≥ 0.5 &nbsp;|&nbsp; 🟠 F1 > 0 &nbsp;|&nbsp; ⚪ F1 = 0
    </p>
    """
    return html

display(HTML(make_html_table(results_df.head(N_PREVIEW))))

Fișier,Keywords generate (mT5),Keywords originale,Precision,Recall,F1
RJP_2014_4_RO_Art-05.pdf,osteopenie; osteoporoză; copil,osteopenie; copil; diagnostic; tratament,0.67,0.50,0.57
RJP_2008_3_Art-03.pdf,sindrom „torch“; infecții congenitale perinatale; copil,sindromul torch; infecții congenitale perinatale,0.67,1.00,0.80
RJP_2008_2_Art-10.pdf,dureri la nivelul membrelor inferioare; copil,dureri ale membrelor inferioare; copil,1.00,1.00,1.00
RMJ_2018_3_Art-07.pdf,hepatită virală c; manifestări hematologice,hepatită virală c; interferon; globule albe; hemoglobină,1.00,0.50,0.67
RJP_2017_2_RO_Art-06.pdf,malformație congenitală de căi aeriene pulmonare; anomalii renale,malformație congenitală pulmonară; chisturi; prognostic,0.50,0.33,0.40
RJP_2014_3_RO_Art-09.pdf,indice intimǎ medie carotidian; ateroscleroză; copil,obezitate; mic pentru vârsta gestaționalǎ; indicele intimă medie carotidian,0.67,0.67,0.67
RMJ_2018_3_Art-08.pdf,intoxicație; alcool etilic; copil,alcool etilic; substanțe de abuz; adolescent; medicamente,0.67,0.50,0.57
RJP_2017_4_RO_Art-03.pdf,insuficiență respiratorie acută; pneumonie interstițială; copil,insuficiență respiratorie; copil; detresă respiratorie acută,1.00,1.00,1.00
RJP_2006_3_Art-08.pdf,artrită idiopatică juvenilă; copil,artrita idiopatică juvenilă; etiologie și fiziopatologie; clasificare; diagnostic diferențial; tratament,0.50,0.20,0.29
RMJ_2019_1_Art-08.pdf,cardiopatie; performanțe; terapie limbaj verbală,limbaj verbal; terapie; monitorizare; test creion-hârtie,0.33,0.25,0.29
